# ETL Reverse Engineering → Data Vault + DBT Pipeline
**Agentic AI-based pipeline for DataStage / Informatica → Snowflake + dbt + datavault4dbt**

## Architecture
```
Input Files (DataStage .dsx / Informatica .xml)
        ↓
[Agent 1] Reverse Engineer  →  Unified Metadata Model
        ↓
[Agent 2] STTM Generator    →  Source-to-Target Mapping (Excel + JSON)
        ↓
[Agent 3] Data Vault Modeler →  Hub / Link / Satellite definitions
        ↓
[Agent 4] DBT Generator     →  dbt models + datavault4dbt macros + packages.yml
        ↓
[Agent 5] Data Profiler     →  ydata-profiling + Presidio PII
        ↓
[Agent 6] Reconciliation    →  Row count / hash / aggregate checks
        ↓
[Agent 7] Data Quality      →  DQ rules + dbt tests + Great Expectations checks
```

## Outputs
| Output | Location |
|---|---|
| Unified metadata JSON | `output/metadata/` |
| STTM Excel + JSON | `output/sttm/` |
| Data Vault model JSON | `output/data_vault/` |
| dbt project (models, schema, packages) | `output/dbt_project/` |
| Profiling HTML report | `output/profiling/` |
| Reconciliation report | `output/reconciliation/` |
| Data quality report | `output/data_quality/` |

## Cell 1 — Install dependencies

In [ ]:
import sys
!{sys.executable} -m pip install -q \
    ydata-profiling presidio-analyzer presidio-anonymizer \
    cryptography faker lxml openpyxl pyyaml jinja2 \
    great-expectations pandas numpy
!{sys.executable} -m spacy download en_core_web_lg
print("✅ Done — restart kernel then run from Cell 2")

## Cell 2 — Configuration

In [ ]:
import warnings, os, json, sys
warnings.filterwarnings("ignore")
sys.path.insert(0, "agents")

# ── INPUT FILES ───────────────────────────────────────────────────────────────
# Replace with your actual file paths
DATASTAGE_FILE    = "input/datastage_sample.dsx"       # or .xml
INFORMATICA_FILE  = "input/informatica_sample.xml"     # or .ipc

# ── TARGET CONFIG ─────────────────────────────────────────────────────────────
TARGET_PLATFORM   = "snowflake"
TARGET_DATABASE   = "ANALYTICS_DB"
TARGET_SCHEMA_RAW = "RAW_VAULT"
TARGET_SCHEMA_BIZ = "BUSINESS_VAULT"
TARGET_SCHEMA_STG = "STAGING"
RECORD_SOURCE     = "ETL_PIPELINE"         # datavault4dbt rsrc value
LOAD_DATE_COLUMN  = "LOAD_DATE"            # datavault4dbt ldts value
HASH_ALGORITHM    = "MD5"                  # MD5 or SHA-256

# ── DBT PROJECT CONFIG ────────────────────────────────────────────────────────
DBT_PROJECT_NAME  = "dv_snowflake_project"
DBT_PROFILE_NAME  = "snowflake_profile"
DATAVAULT4DBT_VERSION = "1.17.0"           # latest as of 2026

# ── OUTPUT DIRS ───────────────────────────────────────────────────────────────
OUTPUT_BASE       = "output"
for d in ["output/metadata","output/sttm","output/data_vault",
          "output/dbt_project/models/staging",
          "output/dbt_project/models/raw_vault",
          "output/dbt_project/models/business_vault",
          "output/dbt_project/tests",
          "output/profiling","output/reconciliation","output/data_quality"]:
    os.makedirs(d, exist_ok=True)

print("✅ Config ready")
print(f"   DataStage  : {DATASTAGE_FILE}")
print(f"   Informatica: {INFORMATICA_FILE}")
print(f"   Target     : {TARGET_PLATFORM.upper()} / {TARGET_DATABASE}")
print(f"   dbt package: datavault4dbt v{DATAVAULT4DBT_VERSION}")

## Cell 3 — Agent 1: Reverse Engineer DataStage + Informatica

In [ ]:
from agents.agent_reverse_engineer import ReverseEngineerAgent

re_agent = ReverseEngineerAgent()

# Parse both files — agent handles whichever exist
metadata = re_agent.run(
    datastage_path=DATASTAGE_FILE   if os.path.exists(DATASTAGE_FILE)   else None,
    informatica_path=INFORMATICA_FILE if os.path.exists(INFORMATICA_FILE) else None,
)

# Save unified metadata
with open("output/metadata/unified_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"✅ Reverse engineering complete")
print(f"   Sources   : {len(metadata.get('sources', []))}")
print(f"   Targets   : {len(metadata.get('targets', []))}")
print(f"   Mappings  : {len(metadata.get('mappings', []))}")
print(f"   Transforms: {len(metadata.get('transformations', []))}")

## Cell 4 — Agent 2: Generate STTM

In [ ]:
from agents.agent_sttm import STTMAgent

sttm_agent = STTMAgent()
sttm = sttm_agent.run(
    metadata=metadata,
    output_dir="output/sttm",
    target_database=TARGET_DATABASE,
    target_schema=TARGET_SCHEMA_RAW,
)

print(f"✅ STTM generated")
print(f"   Mappings : {len(sttm.get('mappings', []))} column mappings")
print(f"   Files    : output/sttm/sttm.xlsx + output/sttm/sttm.json")

## Cell 5 — Agent 3: Data Vault 2.0 Modeling

In [ ]:
from agents.agent_data_vault import DataVaultAgent

dv_agent = DataVaultAgent()
dv_model = dv_agent.run(
    metadata=metadata,
    sttm=sttm,
    output_dir="output/data_vault",
    record_source=RECORD_SOURCE,
    load_date_col=LOAD_DATE_COLUMN,
    hash_algorithm=HASH_ALGORITHM,
)

print(f"✅ Data Vault 2.0 model generated")
print(f"   Hubs       : {len(dv_model.get('hubs', []))}")
print(f"   Links      : {len(dv_model.get('links', []))}")
print(f"   Satellites : {len(dv_model.get('satellites', []))}")

## Cell 6 — Agent 4: Generate dbt Project (datavault4dbt)

In [ ]:
from agents.agent_dbt_generator import DBTGeneratorAgent

dbt_agent = DBTGeneratorAgent()
dbt_result = dbt_agent.run(
    metadata=metadata,
    sttm=sttm,
    dv_model=dv_model,
    output_dir="output/dbt_project",
    project_name=DBT_PROJECT_NAME,
    profile_name=DBT_PROFILE_NAME,
    target_database=TARGET_DATABASE,
    schema_raw=TARGET_SCHEMA_RAW,
    schema_biz=TARGET_SCHEMA_BIZ,
    schema_stg=TARGET_SCHEMA_STG,
    datavault4dbt_version=DATAVAULT4DBT_VERSION,
    hash_algorithm=HASH_ALGORITHM,
    record_source=RECORD_SOURCE,
    load_date_col=LOAD_DATE_COLUMN,
)

print(f"✅ dbt project generated")
print(f"   Staging models   : {dbt_result.get('staging_models', 0)}")
print(f"   Raw vault models : {dbt_result.get('raw_vault_models', 0)}")
print(f"   Schema YAML files: {dbt_result.get('schema_files', 0)}")
print(f"   Location         : output/dbt_project/")

## Cell 7 — Agent 5: Data Profiling (ydata + Presidio PII)

In [ ]:
from agents.agent_profiler import ProfilerAgent
import pandas as pd

# ── Load your actual source data here ────────────────────────────────────────
# df = pd.read_csv("input/your_data.csv")
# df = pd.read_excel("input/your_data.xlsx")

# Synthetic data for demo — replace with your DataFrame
from faker import Faker
import numpy as np, random
fake = Faker(); Faker.seed(42); random.seed(42); np.random.seed(42)
df = pd.DataFrame([{
    "customer_id":  i,
    "full_name":    fake.name(),
    "email":        fake.email(),
    "phone":        fake.phone_number(),
    "ssn":          fake.ssn(),
    "country":      fake.country_code(),
    "revenue":      round(random.uniform(100, 100000), 2),
    "order_count":  random.randint(1, 500),
    "signup_date":  str(fake.date_between(start_date="-5y")),
} for i in range(200)])

profiler_agent = ProfilerAgent()
profiling_result = profiler_agent.run(
    df=df,
    dataset_name="source_customers",
    output_dir="output/profiling",
)

print(f"✅ Profiling complete")
print(f"   PII columns detected : {profiling_result.get('pii_columns', 0)}")
print(f"   Report               : output/profiling/profile_report.html")

## Cell 8 — Agent 6: Data Reconciliation

In [ ]:
from agents.agent_reconciliation import ReconciliationAgent

# ── Provide source and target DataFrames ─────────────────────────────────────
# For demo we simulate a target with minor differences
df_target = df.copy()
df_target.loc[5,  "revenue"] = 0          # simulate a value mismatch
df_target.loc[10, "email"]   = None        # simulate a null introduced
df_target = df_target.iloc[:195]           # simulate 5 missing rows in target

recon_agent = ReconciliationAgent()
recon_result = recon_agent.run(
    df_source=df,
    df_target=df_target,
    key_column="customer_id",
    numeric_cols=["revenue", "order_count"],
    output_dir="output/reconciliation",
)

print(f"✅ Reconciliation complete")
print(f"   Row count match   : {recon_result.get('row_count_match')}")
print(f"   Missing in target : {recon_result.get('missing_in_target')}")
print(f"   Value mismatches  : {recon_result.get('value_mismatches')}")
print(f"   Report            : output/reconciliation/recon_report.xlsx")

## Cell 9 — Agent 7: Data Quality

In [ ]:
from agents.agent_data_quality import DataQualityAgent

dq_agent = DataQualityAgent()
dq_result = dq_agent.run(
    df=df,
    metadata=metadata,
    sttm=sttm,
    dv_model=dv_model,
    output_dir="output/data_quality",
    dbt_project_dir="output/dbt_project",
)

print(f"✅ Data Quality complete")
print(f"   DQ rules generated  : {dq_result.get('rules_generated', 0)}")
print(f"   dbt tests written   : {dq_result.get('dbt_tests', 0)}")
print(f"   Rules passed        : {dq_result.get('rules_passed', 0)}")
print(f"   Rules failed        : {dq_result.get('rules_failed', 0)}")
print(f"   Report              : output/data_quality/dq_report.xlsx")

## Cell 10 — Pipeline summary

In [ ]:
import pandas as pd
from IPython.display import display

summary = pd.DataFrame([
    {"Agent": "1 - Reverse Engineer",  "Status": "✅", "Output": "output/metadata/unified_metadata.json"},
    {"Agent": "2 - STTM",              "Status": "✅", "Output": "output/sttm/sttm.xlsx + sttm.json"},
    {"Agent": "3 - Data Vault 2.0",    "Status": "✅", "Output": "output/data_vault/dv_model.json"},
    {"Agent": "4 - DBT Generator",     "Status": "✅", "Output": "output/dbt_project/"},
    {"Agent": "5 - Data Profiler",     "Status": "✅", "Output": "output/profiling/profile_report.html"},
    {"Agent": "6 - Reconciliation",    "Status": "✅", "Output": "output/reconciliation/recon_report.xlsx"},
    {"Agent": "7 - Data Quality",      "Status": "✅", "Output": "output/data_quality/dq_report.xlsx"},
])
display(summary)
print("\n🎉 Full pipeline complete!")